# 06 — Accessibility prediction: baseline classifier

Predicting whether an edge is likely wheelchair-accessible from its own attributes (highway type, surface, street_count), using the real tagged edges as labeled data. Baseline only — no graph-neighbor information yet (that's the GNN's job, next notebook).

### Download the area (reusing the cached data)

Same Downtown Toronto graph as notebooks 04/05.

In [1]:
import sys
sys.path.append("..")

import osmnx as ox
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

ox.settings.cache_folder = "cache"
ox.settings.useful_tags_way += ["wheelchair", "surface"]
ox.settings.useful_tags_node += ["kerb", "crossing", "tactile_paving"]
ox.settings.overpass_url = "https://overpass.kumi.systems/api/interpreter"

G_big = ox.graph_from_place("Downtown Toronto, Toronto, Ontario, Canada", network_type="walk")
print(G_big.number_of_edges(), "edges")

39170 edges


### Collect labeled edges into a table

Only edges with a real `wheelchair` tag (`yes`/`designated` → 1, `no` → 0) become labeled rows — the other ~38,800 untagged edges are exactly what this model is meant to predict for later, so they can't be used as training labels now. Features per edge: `length`, `highway` type, `surface` (or `"unknown"` if missing), and `street_count` — which lives on the node, not the edge, so it's pulled from `G_big.nodes[u]`.

In [2]:
rows = []

for u, v, k, data in G_big.edges(keys=True, data=True):
    wheelchair = data.get("wheelchair")
    if wheelchair not in ("yes", "designated", "no"):
        continue

    highway = data.get("highway", "unknown")
    if isinstance(highway, list):
        highway = highway[0]

    surface = data.get("surface", "unknown")
    if isinstance(surface, list):
        surface = surface[0]

    street_count = G_big.nodes[u].get("street_count", 0)
    x = G_big.nodes[u].get("x", 0)

    rows.append({
        "u": u, "v": v, "k": k,
        "length": data.get("length", 0),
        "highway": highway,
        "surface": surface,
        "street_count": street_count,
        "x": x,
        "label": 1 if wheelchair in ("yes", "designated") else 0,
    })

df = pd.DataFrame(rows)
print(df.shape)
print(df["label"].value_counts())
df.head()

(174, 9)
label
1    158
0     16
Name: count, dtype: int64


,u,v,k,length,highway,surface,street_count,x,label
0,391006074,11522441209,0,53.657287,footway,unknown,3,-79.389403,1
1,394502563,11396046466,0,9.923720,footway,concrete,4,-79.393442,1
2,400707035,516949102,0,30.927468,footway,unknown,3,-79.355473,0
3,419511967,419511968,0,9.349883,footway,concrete,4,-79.392578,0
4,419511968,419511967,0,9.349883,footway,concrete,4,-79.392608,0


### Geography-based train/test split

Not a random shuffle — split on each edge's longitude (`x` of node `u`) at the median, so train and test edges come from genuinely different halves of the area. This is a stricter, more honest test than a random split: it checks whether the model generalizes to a part of the city it never saw, not just to unseen edges that happen to be right next to ones it trained on.

In [3]:
median_x = df["x"].median()
train_df = df[df["x"] < median_x]
test_df = df[df["x"] >= median_x]

print("train:", len(train_df), " test:", len(test_df))

feature_cols = ["length", "street_count"]
categorical_cols = ["highway", "surface"]

train_encoded = pd.get_dummies(train_df[feature_cols + categorical_cols], columns=categorical_cols)
test_encoded = pd.get_dummies(test_df[feature_cols + categorical_cols], columns=categorical_cols)

# align columns in case train/test saw different highway/surface categories
test_encoded = test_encoded.reindex(columns=train_encoded.columns, fill_value=0)

X_train, y_train = train_encoded, train_df["label"]
X_test, y_test = test_encoded, test_df["label"]

train: 87  test: 87


### Train and evaluate the baseline

Same `RandomForestClassifier` pattern from the toy example, now on the real, geographically-split data.

In [6]:
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
model.fit(X_train, y_train)

predictions = model.predict(X_test)
print("accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

accuracy: 0.7471264367816092
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         6
           1       0.92      0.80      0.86        81

    accuracy                           0.75        87
   macro avg       0.46      0.40      0.43        87
weighted avg       0.85      0.75      0.80        87



In [5]:
print("train labels:")
print(train_df["label"].value_counts())
print()
print("test labels:")
print(test_df["label"].value_counts())

train labels:
label
1    77
0    10
Name: count, dtype: int64

test labels:
label
1    81
0     6
Name: count, dtype: int64
